# 09 — Active Cache and Data Subscriptions

Pull model: `data_formula` with `_cache_time` for periodic refresh, `data.subscribe` for side effects.

**What you learn:**
- `data_formula` with `_cache_time=-N`: active cache with background refresh (requires async)
- `data.subscribe()`: react to data changes with side effects (logging, file writes)
- Pull model: formulas compute on-demand, active cache pushes periodic updates
- Side effects belong to the manager/application, not to the builder

**Prerequisites:** 08_reactive_basics

In [ ]:
from pathlib import Path
from IPython.display import HTML

from genro_bag import Bag
from genro_bag.resolvers import FileResolver
from genro_builders.builder import component
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import ReactiveManager
from genro_toolbox import set_sync

set_sync()

In [ ]:
class DashBuilder(HtmlBuilder):
    @component(main_tag='div')
    def gauge(self, comp, label_text=None, value=None, unit=None, **kwargs):
        comp.div(label_text, _class='label')
        comp.div(value, _class='value')
        comp.div(unit, _class='unit')

class Dashboard(ReactiveManager):
    def on_init(self):
        self.page = self.register_builder('page', DashBuilder)

    def main(self, source):
        # FileResolver with cache_time=5: the CSS file is read once and
        # cached for 5 seconds. Subsequent renders within that window
        # reuse the cached content. After 5s, the next render re-reads
        # the file — handy during development when you edit the CSS.
        # (cache_time=0 would re-read on every render;
        #  cache_time=-5 would refresh in background every 5s, async only)
        source.head().style(FileResolver('dashboard.css', cache_time=5))
        body = source.body()
        body.h1('DASHBOARD')
        dash = body.div(_class='dash', datapath='gauges')

        dash.gauge(_class='gauge speed', label_text='SPEED', value='^.speed', unit='km/h')
        dash.gauge(_class='gauge rpm', label_text='RPM', value='^.rpm', unit='x1000')
        dash.gauge(_class='gauge oil', label_text='OIL TEMP', value='^.oil_temp', unit='C')
        dash.gauge(_class='gauge fuel', label_text='FUEL', value='^.fuel', unit='%')

In [ ]:
app = Dashboard()
app.local_store('page')['gauges'] = Bag.from_json(
    '{"speed": 87, "rpm": 3250, "oil_temp": 92, "fuel": 73}',
)
app.run(subscribe=True)

app.global_store.subscribe(
    'logger',
    any=lambda pathlist=None, **kw: print(f"  changed: {'.'.join(str(p) for p in pathlist)}") if pathlist else None,
)

HTML(app.page.render())

## Update gauges manually

In pull model, update data and re-render explicitly.

In [ ]:
store = app.local_store('page')
store['gauges.speed'] = 120
store['gauges.rpm'] = 4500
print(f"Speed: {store['gauges.speed']}, RPM: {store['gauges.rpm']}")
HTML(app.page.render())